<a href="https://colab.research.google.com/github/Shahmir-028/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahmir-028/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds the rule baseline for my lane (**content refresh prioritization**). It checks two signals, encodes one transparent rule, writes a ranked queue to `work/outputs/baseline_action_score.csv`, and reviews the top ten rows with a skeptic's eye. This baseline is what my Week-5 model has to beat.

**Outcome used for checking.** A page counts as *declining* when `trend_direction == "down"` (impressions in the last 30 days are more than 20% below the 30 days before). The outcome is only used to *check* signals and to *score* the ranking. It is never an input to the rule.

Nothing here is random, so Run all gives the same queue every time. Wording is careful on purpose: everything below is an observed, directional, decision-support pattern, not proof that a refresh causes a recovery.

## 1. My rule and its reason codes

**The rule in plain words.** Send a page to refresh review when (1) it has not been updated for 91 days or more, (2) it had real search demand in days 31–90 back (500+ impressions), and (3) it got fewer clicks than a typical page at the same position. Rank the flagged pages by how many clicks they are missing.

**Reason code (only one).** `stale_and_below_position_norm`. Pages the rule does not flag get `not_flagged`.

**Action label (only one).** `refresh_and_fix_ctr`: refresh the content and re-check the title and description. Pages the rule does not flag get `monitor`.

**What the rule is allowed to read:** `days_since_last_update`, `avg_position`, and impressions and clicks from days 31–90 back (the 90-day total minus the most recent 30 days). **What it never reads:** `trend_direction`, `trend_pct`, `is_declining_label`, or anything from the most recent 30 days.

Before I encode it, I check the two signals it leans on. Both sit behind real FlyRank flags from the session: **staleness** (behind the refresh flags) and **CTR vs position** (behind the CTR-fix logic).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_NAME = "content_refresh_anonymized.csv"
RAW_URL = ("https://raw.githubusercontent.com/Shahmir-028/"
           "flyrank-ml-internship/main/data/raw/" + DATA_NAME)

# find the repo root: works from the repo root, from work/notebooks/, or in Colab
if Path("data/raw", DATA_NAME).exists():
    ROOT, SOURCE = Path("."), Path("data/raw", DATA_NAME)
elif Path("../../data/raw", DATA_NAME).exists():
    ROOT, SOURCE = Path("../.."), Path("../../data/raw", DATA_NAME)
elif Path("/content", DATA_NAME).exists():
    ROOT, SOURCE = Path("."), Path("/content", DATA_NAME)
else:
    ROOT, SOURCE = Path("."), RAW_URL

OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SOURCE)
print("rows, columns:", df.shape)
assert df.shape == (30000, 44)

# Outcome: used ONLY to check the signals and to score the baseline. Never a rule input.
df["declining"] = (df["trend_direction"] == "down").astype(int)

# Days 31-90 back = the 90-day total minus the most recent 30 days.
# The most recent 30 days are the outcome window, so they stay out of the rule.
df["pre_impressions"] = df["impressions_90d"] - df["impressions_last_30d"]
df["pre_clicks"] = df["clicks_90d"] - df["clicks_last_30d"]
assert (df["pre_impressions"] >= 0).all() and (df["pre_clicks"] >= 0).all()

STALE_DAYS = 91        # "stale" = not updated for 91+ days
VOLUME_FLOOR = 500     # need 500+ impressions in days 31-90 back, or CTR is too noisy to read
POS_BINS = [0, 3, 6, 10, 20, 50, 1000]
POS_LABELS = ["1-3", "3.1-6", "6.1-10", "10.1-20", "20.1-50", "50+"]
df["pos_bucket"] = pd.cut(df["avg_position"], bins=POS_BINS, labels=POS_LABELS)  # position 0 = no data -> NaN

base_rate = df["declining"].mean()
print(f"Pages: {len(df):,} | declining (outcome): {df['declining'].sum():,} = {base_rate:.1%}")
print(f"Pages with no position data (avg_position = 0): {(df['avg_position'] == 0).sum():,}")


def rate_table(frame, by):
    """Decline rate per bucket, with n and a rough 95% margin so small buckets are not over-read."""
    g = frame.groupby(by, observed=True)["declining"].agg(n="size", declining_rate="mean")
    g["+/-95%"] = 1.96 * np.sqrt(g["declining_rate"] * (1 - g["declining_rate"]) / g["n"])
    return g.round(3)

rows, columns: (30000, 44)
Pages: 30,000 | declining (outcome): 16,262 = 54.2%
Pages with no position data (avg_position = 0): 1,205


### Signal 1: staleness (behind the refresh flags)

**What the flag assumes:** pages that have gone longer without an update lose search traffic more often.

**My test:** decline rate by days since last update, with n and a rough 95% margin. Then stale vs fresh inside each content-age tier (so I don't confuse "stale" with "old"), and inside each client (so I don't confuse "stale" with "which client the page belongs to").

In [ ]:
# SIGNAL 1 - staleness: do pages that were not updated for longer decline more?
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=[0, 30, 90, 180, 10_000],
                            labels=["0-30", "31-90", "91-180", "181+"])
print("Decline rate by days since last update (all pages):")
print(rate_table(df, "stale_bucket").to_string())
print(f"\nBase rate: {base_rate:.3f}")

print("\nMost common values of days_since_last_update (n pages):")
print(df["days_since_last_update"].value_counts().head(4).to_string())

df["is_stale"] = df["days_since_last_update"] >= STALE_DAYS
n_stale, n_fresh = int(df["is_stale"].sum()), int((~df["is_stale"]).sum())
rate_stale = df.loc[df["is_stale"], "declining"].mean()
rate_fresh = df.loc[~df["is_stale"], "declining"].mean()
pooled_gap = rate_stale - rate_fresh
print(f"\nStale (91+ days): n={n_stale:,}, decline rate {rate_stale:.3f}")
print(f"Fresh (<91 days): n={n_fresh:,}, decline rate {rate_fresh:.3f}")
print(f"Pooled gap (stale minus fresh): {pooled_gap:+.3f}")

print("\nStale vs fresh inside each content-age tier:")
tier = (df.groupby(["age_tier", "is_stale"], observed=True)["declining"]
          .agg(n="size", rate="mean").round(3).unstack("is_stale"))
tier.columns = [f"{a}_{'stale' if b else 'fresh'}" for a, b in tier.columns]
print(tier.to_string())

# Is it the page, or just which client the page belongs to? Compare inside each client.
rows = []
for client, part in df.groupby("client_id"):
    s, f = part[part["is_stale"]], part[~part["is_stale"]]
    if len(s) >= 30 and len(f) >= 30:
        rows.append((len(s) + len(f), s["declining"].mean() - f["declining"].mean()))
within = pd.DataFrame(rows, columns=["pages", "stale_minus_fresh"])
stale_higher = int((within["stale_minus_fresh"] > 0).sum())
within_gap = float(np.average(within["stale_minus_fresh"], weights=within["pages"]))
print(f"\nClients with 30+ pages on both sides: {len(within)}")
print(f"  stale pages decline MORE in {stale_higher} of {len(within)} of them")
print(f"  page-weighted average gap inside clients: {within_gap:+.3f}")

Decline rate by days since last update (all pages):
                  n  declining_rate  +/-95%
stale_bucket                               
0-30          20480           0.511   0.007
31-90           175           0.589   0.073
91-180         9171           0.611   0.010
181+            174           0.471   0.074

Base rate: 0.542

Most common values of days_since_last_update (n pages):
days_since_last_update
20     11573
104     8773
22      3564
8       1929

Stale (91+ days): n=9,345, decline rate 0.608
Fresh (<91 days): n=20,655, decline rate 0.512
Pooled gap (stale minus fresh): +0.096

Stale vs fresh inside each content-age tier:
          n_fresh  n_stale  rate_fresh  rate_stale
age_tier                                          
181-365    4767.0   6601.0       0.417       0.586
31-90       492.0      NaN       0.669         NaN
365+       5820.0    540.0       0.419       0.506
91-180     9576.0   2204.0       0.608       0.701

Clients with 30+ pages on both sides: 13
  stale

**Verdict: MIXED.**

The direction is right. Pages stale for 91+ days decline 60.8% of the time (n = 9,345) against 51.2% for fresher pages (n = 20,655). The gap shows up in every age tier that has enough pages.

But it is weaker than it first looks:
- The gap shrinks from +9.6 points pooled to +5.3 points *inside* clients, so about half of it is client mix.
- Stale pages decline more in only 9 of the 13 clients I can compare. The other 4 point the other way.
- The 31–90 and 181+ buckets have about 175 pages each (margin about ±7 points), so I can't read them. The 181+ bucket even sits below the base rate.
- The field is lumpy. Most pages sit at exactly 20 or exactly 104 days. That looks like shared update dates, not page-by-page neglect.

So staleness is a real but modest signal. It was worth testing: it told me not to lean on it too hard.

### Signal 2: CTR vs position (behind the CTR-fix logic)

**What the flag assumes:** at a given position a page should win roughly the typical share of clicks, so a page far below that is the one in trouble.

**My test:** first, what is the typical CTR at each position (only pages with 500+ impressions, rates are in %, so 0.40 means 0.40%; position 0 means "no data" and is left out)? Then the decline rate by how a page's clicks compare with the typical clicks for its position. I repeat it inside clients and without zero-click pages.

In [ ]:
# SIGNAL 2 - CTR vs position: is a page that gets fewer clicks than its position "should" a page in trouble?
known = df[df["pos_bucket"].notna() & (df["pre_impressions"] >= VOLUME_FLOOR)].copy()
print(f"Pages with a known position and {VOLUME_FLOOR}+ impressions in days 31-90 back: {len(known):,}")

# Step A: what is a "typical" CTR at each position? (pooled clicks / pooled impressions, no labels used)
pos_table = known.groupby("pos_bucket", observed=True).agg(
    n=("pre_clicks", "size"), clicks=("pre_clicks", "sum"), impressions=("pre_impressions", "sum"))
ctr_norm = pos_table["clicks"] / pos_table["impressions"]          # a fraction, not x100
known["ctr_pct"] = known["pre_clicks"] / known["pre_impressions"] * 100
pos_table["typical_ctr_pct"] = (ctr_norm * 100).round(3)
pos_table["median_ctr_pct"] = known.groupby("pos_bucket", observed=True)["ctr_pct"].median().round(3)
print("\nCTR by position (rates are in %, so 0.40 means 0.40%):")
print(pos_table[["n", "typical_ctr_pct", "median_ctr_pct"]].to_string())

# Step B: how many clicks did each page get compared with a typical page at its position?
known["expected_clicks"] = known["pre_impressions"] * known["pos_bucket"].map(ctr_norm).astype(float)
known["click_ratio"] = known["pre_clicks"] / known["expected_clicks"]
known["ratio_bucket"] = pd.cut(known["click_ratio"], bins=[-0.01, 0.25, 0.5, 1.0, 2.0, np.inf],
                               labels=["<0.25x", "0.25-0.5x", "0.5-1x", "1-2x", ">2x"])
ratio_table = rate_table(known, "ratio_bucket")
ratio_table["zero_click_share"] = (known.groupby("ratio_bucket", observed=True)["pre_clicks"]
                                   .apply(lambda s: (s == 0).mean()).round(3))
eligible_rate = known["declining"].mean()
print("\nDecline rate by actual clicks / typical clicks for the page's position:")
print(ratio_table.to_string())

# zero-click pages could be tracking gaps, so repeat the table without them
print("\nSame table, only pages with at least one click:")
print(rate_table(known[known["pre_clicks"] > 0], "ratio_bucket").to_string())
print(f"\nBase rate of these pages: {eligible_rate:.3f} (all pages: {base_rate:.3f})")

# Step C: same comparison inside each client (low = under 0.5x, high = 1x or more)
rows = []
for client, part in known.groupby("client_id"):
    low, high = part[part["click_ratio"] < 0.5], part[part["click_ratio"] >= 1.0]
    if len(low) >= 30 and len(high) >= 30:
        rows.append((len(low) + len(high), low["declining"].mean() - high["declining"].mean()))
within2 = pd.DataFrame(rows, columns=["pages", "low_minus_high"])
low_higher = int((within2["low_minus_high"] > 0).sum())
within2_gap = float(np.average(within2["low_minus_high"], weights=within2["pages"]))
print(f"\nClients with 30+ pages in both the low and high group: {len(within2)}")
print(f"  low-CTR pages decline MORE in {low_higher} of {len(within2)} of them")
print(f"  page-weighted average gap inside clients: {within2_gap:+.3f}")

Pages with a known position and 500+ impressions in days 31-90 back: 15,441

CTR by position (rates are in %, so 0.40 means 0.40%):
               n  typical_ctr_pct  median_ctr_pct
pos_bucket                                       
1-3          467            0.460           0.183
3.1-6       2892            0.397           0.279
6.1-10      3874            0.272           0.190
10.1-20     4048            0.330           0.158
20.1-50     3892            0.135           0.076
50+          268            0.046           0.000

Decline rate by actual clicks / typical clicks for the page's position:
                 n  declining_rate  +/-95%  zero_click_share
ratio_bucket                                                
<0.25x        4190           0.711   0.014             0.634
0.25-0.5x     2654           0.657   0.018             0.000
0.5-1x        3680           0.609   0.016             0.000
1-2x          3056           0.550   0.018             0.000
>2x           1861           

**Verdict: CONFIRMED.**

Pages with under a quarter of the typical clicks for their position decline 71.1% of the time (n = 4,190). Pages with more than twice the typical clicks decline 47.1% (n = 1,861). The buckets in between step down in order: 65.7%, 60.9%, 55.0%. The same direction holds in 13 of 13 clients with enough pages (+19.3 points inside clients), and it still holds when I drop the zero-click pages (lowest bucket 75.3%, n = 1,533).

Two honest caveats:
- The yardstick is rough. Typical CTR at positions 10.1–20 (0.330%) is above 6.1–10 (0.272%), and the median at positions 1–3 (0.183%) is below 3.1–6 (0.279%). CTR levels in this slice are small and the top tier is thin (n = 467).
- 63% of the lowest bucket have zero clicks. Some of those may be tracking gaps rather than weak pages.

This is an observed association. It does not show that fixing CTR stops a decline.

**What the two verdicts do to my rule.**
- CTR vs position was CONFIRMED, so the score is built from it: the number of clicks a page is missing compared with a typical page at its position.
- Staleness was MIXED, so it stays only as a **gate** (don't queue a page that was updated recently), with no extra weight. I do not multiply the score by days stale.
- The 500-impression floor stays, because the data dictionary warns that CTR on thin volume is unreadable.

## 2. Build the ranked queue (writes the CSV)

**Score = missing clicks** = `max(expected clicks - actual clicks, 0)`, where expected clicks = impressions in days 31–90 back x the typical CTR at the page's position bucket. No weights are fitted, so a human can recompute any row by hand.

A page gets a score only if it is stale (91+ days), has 500+ impressions in that window, has a known position, and is below the typical clicks. Every other page scores 0 and is labelled `monitor`. Ties are broken by volume, then `content_id`, so the order is the same on every run.

The CSV is regenerated on every run and stays out of git (the CI leak-guard blocks data files). What I commit is this notebook and the small metrics JSON, which are the receipts for the numbers.

In [ ]:
REASON_CODE = "stale_and_below_position_norm"
ACTION_LABEL = "refresh_and_fix_ctr"
RULE_INPUTS = ["days_since_last_update", "pre_impressions", "pre_clicks", "avg_position"]


def baseline_action_score(d, ctr_norm):
    """Score = clicks the page is missing compared with a typical page at its position.
    Counted only if the page is stale and has enough impressions to trust. No fitted weights."""
    bucket = pd.cut(d["avg_position"], bins=POS_BINS, labels=POS_LABELS)
    expected_clicks = d["pre_impressions"] * bucket.map(ctr_norm).astype(float)
    click_gap = (expected_clicks - d["pre_clicks"]).clip(lower=0)
    flagged = (
        (d["days_since_last_update"] >= STALE_DAYS)
        & (d["pre_impressions"] >= VOLUME_FLOOR)
        & bucket.notna()
        & (click_gap > 0)
    )
    return pd.DataFrame({
        "score": np.where(flagged, click_gap, 0.0),
        "expected_clicks": expected_clicks,
        "reason_code": np.where(flagged, REASON_CODE, "not_flagged"),
        "action": np.where(flagged, ACTION_LABEL, "monitor"),
    }, index=d.index)


scored = pd.concat([df, baseline_action_score(df, ctr_norm)], axis=1)
# ties: bigger pre-window volume first, then content_id, so the ranking is the same on every run
scored = scored.sort_values(["score", "pre_impressions", "content_id"],
                            ascending=[False, False, True], kind="mergesort").reset_index(drop=True)
scored["rank"] = np.arange(1, len(scored) + 1)

QUEUE_COLUMNS = ["rank", "content_id", "client_id", "score", "reason_code", "action",
                 "expected_clicks", "pre_clicks", "pre_impressions", "avg_position",
                 "days_since_last_update", "content_age_days", "content_type"]
scored[QUEUE_COLUMNS].round(3).to_csv(OUT_DIR / "baseline_action_score.csv", index=False)

flagged_rows = scored[scored["reason_code"] == REASON_CODE]
print(f"Ranked {len(scored):,} pages -> wrote work/outputs/baseline_action_score.csv")
print(f"Flagged by the rule: {len(flagged_rows):,} ({len(flagged_rows) / len(scored):.1%} of pages)")
print("\nReason codes in the queue:")
print(scored["reason_code"].value_counts().to_string())
print("\nActions in the queue:")
print(scored["action"].value_counts().to_string())

Ranked 30,000 pages -> wrote work/outputs/baseline_action_score.csv
Flagged by the rule: 4,338 (14.5% of pages)

Reason codes in the queue:
reason_code
not_flagged                      25662
stale_and_below_position_norm     4338

Actions in the queue:
action
monitor                25662
refresh_and_fix_ctr     4338


In [ ]:
# How good is the baseline? precision@K = of the top K pages, how many were actually declining.
# (The outcome is only used here, to score the ranking. It never fed the rule.)
pool = scored[scored["pre_impressions"].ge(VOLUME_FLOOR) & scored["avg_position"].gt(0)]
pool_rate = pool["declining"].mean()
flagged_rate = flagged_rows["declining"].mean()

prec_rows = []
for k in [10, 50, 100, 500, 1000]:
    p = scored.head(k)["declining"].mean()
    prec_rows.append({"K": k, "precision_at_K": round(p, 3), "base_rate_all_pages": round(base_rate, 3),
                 "base_rate_rule_pool": round(pool_rate, 3), "lift_vs_pool": round(p / pool_rate, 2)})
prec = pd.DataFrame(prec_rows)
print(prec.to_string(index=False))
print(f"\nAll {len(flagged_rows):,} flagged pages: {flagged_rate:.3f} declining "
      f"(rule pool {pool_rate:.3f}, all pages {base_rate:.3f})")

from sklearn.dummy import DummyClassifier   # the floor below the floor
dummy = DummyClassifier(strategy="most_frequent").fit(df[["pre_impressions"]], df["declining"])
print(f"Dummy floor (always say 'declining'): accuracy, same as the base rate, {dummy.score(df[['pre_impressions']], df['declining']):.3f}")

   K  precision_at_K  base_rate_all_pages  base_rate_rule_pool  lift_vs_pool
  10           0.700                0.542                0.617          1.14
  50           0.580                0.542                0.617          0.94
 100           0.570                0.542                0.617          0.92
 500           0.640                0.542                0.617          1.04
1000           0.662                0.542                0.617          1.07

All 4,338 flagged pages: 0.683 declining (rule pool 0.617, all pages 0.542)
Dummy floor (always say 'declining'): accuracy, same as the base rate, 0.542


**Reading the numbers (decision-support, not proof).**
- Of all 30,000 pages, 54.2% decline. Among pages the rule is *allowed* to flag (500+ impressions, known position) the rate is 61.7%. Among the 4,338 pages it actually flags it is 68.3%. So the flag itself is directionally useful (+6.6 points over its own pool).
- Inside the queue, ranking by missing clicks does **not** help at the very top: precision@50 is 0.58 and precision@100 is 0.57, both *below* the pool rate of 0.617. Precision@10 is 0.70, but with ten pages one page is ten points.
- The reason is that the score counts raw missing clicks, so the biggest pages win whatever their chance of declining. Section 4 shows what that does to the top of the list.

These are the numbers the Week-5 model must beat: the flagged-set rate (0.683) and precision@K above, measured on this same data and outcome. The dummy floor (always say "declining") is the base rate, 0.542.

## 3. Top-10 review

*For each of my top ten: the action, why it is there, and what would make it wrong.* All ten carry the same action (`refresh_and_fix_ctr`) and reason code (`stale_and_below_position_norm`). Pages are shown by the last six characters of their pseudonymous id. CTR numbers are in %, over days 31–90 back, and "missing" means the score (clicks a typical page at that position would have won).

In [ ]:
top10 = scored.head(10).copy()
top10["page"] = top10["content_id"].str[-6:]      # short pseudonymous tag, not the full id
top10["client"] = top10["client_id"].str[-4:]
top10["ctr_pct"] = top10["pre_clicks"] / top10["pre_impressions"] * 100
top10["typical_ctr_pct"] = top10["expected_clicks"] / top10["pre_impressions"] * 100
show = top10[["rank", "page", "client", "action", "score", "pre_impressions", "pre_clicks",
              "ctr_pct", "typical_ctr_pct", "avg_position", "days_since_last_update",
              "content_age_days", "content_type"]]
print(show.round(2).to_string(index=False))

 rank   page client              action   score  pre_impressions  pre_clicks  ctr_pct  typical_ctr_pct  avg_position  days_since_last_update  content_age_days    content_type
    1 04994d   8562 refresh_and_fix_ctr 1052.88           396924         521     0.13             0.40           4.2                     104               537 keyword article
    2 ce36be   27de refresh_and_fix_ctr  583.51           237442         358     0.15             0.40           5.6                     104               126 keyword article
    3 ab9013   27de refresh_and_fix_ctr  395.56           145352           0     0.00             0.27           9.7                     104               362 keyword article
    4 66c305   27de refresh_and_fix_ctr  393.27           137765         153     0.11             0.40           5.7                     104               257 keyword article
    5 c8214e   27de refresh_and_fix_ctr  392.93           188112         119     0.06             0.27           7.3         

1. **#1 `…04994d`, refresh_and_fix_ctr.** *Why:* 397k impressions at position 4.2 but only 521 clicks (0.13% vs about 0.40% typical there), 104 days since update; about 1,050 clicks missing. *Wrong if:* the results page itself answers the query (an answer box or AI summary), so clicks stay low whatever the title says.
2. **#2 `…ce36be`, refresh_and_fix_ctr.** *Why:* 237k impressions at position 5.6, 358 clicks (0.15% vs 0.40%), 104 days since update; about 580 missing. *Wrong if:* the page is simply young (126 days old, so it was last touched only ~3 weeks after launch) and is still settling, which makes a rewrite premature.
3. **#3 `…ab9013`, refresh_and_fix_ctr.** *Why:* 145k impressions at position 9.7 and zero clicks (typical is 0.27%); about 400 missing. *Wrong if:* the zero is a data gap (clicks not tracked for this URL) and not a weak page. 145k impressions with no click at all is suspicious.
4. **#4 `…66c305`, refresh_and_fix_ctr.** *Why:* 138k impressions at position 5.7, 153 clicks (0.11% vs 0.40%); about 390 missing. *Wrong if:* many impressions come from loosely related queries the page doesn't really answer, so low CTR means topic mismatch and a title tweak won't fix it.
5. **#5 `…c8214e`, refresh_and_fix_ctr.** *Why:* 188k impressions at position 7.3, 119 clicks (0.06% vs 0.27%); about 390 missing. *Wrong if:* the 7.3 average hides big swings (page 1 on some days, page 3 on others), so "typical CTR at 7.3" is the wrong yardstick for it.
6. **#6 `…2c3a35`, refresh_and_fix_ctr.** *Why:* 153k impressions at position 4.3, 359 clicks (0.23% vs 0.40%); about 250 missing. *Wrong if:* 0.23% is inside the normal spread. It is only about 0.6x of "typical", and the median page at positions 3.1–6 gets 0.28%, so this page may be ordinary and only look weak because the typical value is weighted toward big, high-CTR pages.
7. **#7 `…551e2c`, refresh_and_fix_ctr.** *Why:* 79k impressions at position 4.8, 66 clicks (0.08% vs 0.40%); about 250 missing. *Wrong if:* this client's pages are searched or measured differently (it is the only page from its client in my top ten), so a norm built across all clients doesn't fit it.
8. **#8 `…af3a3d`, refresh_and_fix_ctr.** *Why:* 118k impressions at position 6.4, 76 clicks (0.06% vs 0.27%); about 245 missing. *Wrong if:* "104 days since update" is a shared bulk date and not a real edit history (98 of my top 100 carry exactly 104), so the page may already have been edited and isn't stale at all.
9. **#9 `…fec61a`, refresh_and_fix_ctr.** *Why:* 77k impressions at position 3.8, 66 clicks (0.09% vs 0.40%), 445 days old; about 240 missing. *Wrong if:* the impressions come from a short spike (a season or a news burst) inside the window, so the click gap is an event and not steady underperformance.
10. **#10 `…d13b4a`, refresh_and_fix_ctr.** *Why:* 70k impressions at position 3.4, 46 clicks (0.07% vs 0.40%); about 230 missing. *Wrong if:* part of those impressions come from rank-tracking tools or bots that never click, which inflates impressions and drags CTR down with no real readers behind it.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Then confirm no label-derived columns, future windows, or product flags leaked in.* The outcome check below is a fair look, because the outcome never fed the rule.

In [ ]:
# Outcome check on the top of the queue (the outcome never fed the rule, so this is a fair look)
top10 = scored.head(10).copy()
top100 = scored.head(100).copy()
top10["page"] = top10["content_id"].str[-6:]

print("Top 10: outcome of each pick")
print(top10[["rank", "page", "trend_direction", "declining"]].to_string(index=False))
print(f"\nTop 10 declining: {int(top10['declining'].sum())} of 10 | top 100: {int(top100['declining'].sum())} of 100")

print("\nWhere the top 100 comes from (client tag = last 4 characters of the pseudonym):")
print(top100["client_id"].str[-4:].value_counts().head(4).to_string())
biggest_client_share = top100["client_id"].value_counts(normalize=True).iloc[0]

zero_click_top10 = int((top10["pre_clicks"] == 0).sum())
zero_click_top100 = int((top100["pre_clicks"] == 0).sum())
print(f"\nZero clicks despite big impressions: {zero_click_top10} of top 10, {zero_click_top100} of top 100")

print("\nMost common days_since_last_update among all flagged pages:")
print(flagged_rows["days_since_last_update"].value_counts().head(3).to_string())
same_day_top100 = int((top100["days_since_last_update"] == flagged_rows["days_since_last_update"].mode().iloc[0]).sum())
print(f"Top 100 pages sharing that exact value: {same_day_top100}")

young_top100 = int((top100["content_age_days"] < 180).sum())
print(f"Top 100 pages younger than 180 days: {young_top100}")

Top 10: outcome of each pick
 rank   page trend_direction  declining
    1 04994d            down          1
    2 ce36be            down          1
    3 ab9013            down          1
    4 66c305          stable          0
    5 c8214e          stable          0
    6 2c3a35            down          1
    7 551e2c            down          1
    8 af3a3d            down          1
    9 fec61a          stable          0
   10 d13b4a            down          1

Top 10 declining: 7 of 10 | top 100: 57 of 100

Where the top 100 comes from (client tag = last 4 characters of the pseudonym):
client_id
27de    65
0f77    25
8562     5
5f04     2

Zero clicks despite big impressions: 1 of top 10, 2 of top 100

Most common days_since_last_update among all flagged pages:
days_since_last_update
104    4261
106      41
92        9
Top 100 pages sharing that exact value: 98
Top 100 pages younger than 180 days: 34


**Weak picks, in order of how much they worry me:**
1. **The top of the queue is not better than the flag.** 7 of my top 10 were declining; ranks 4, 5 and 9 were `stable`. Across the top 100 it is 57 of 100, below the 61.7% of the rule's pool. The order inside the queue is driven by volume, not by likelihood of decline.
2. **It is mostly one or two clients.** 65 of the top 100 come from a single client tag and 25 from another. Raw missing clicks favor big sites with big pages. A content team that works client by client would want one queue per client.
3. **"Stale" is mostly one date.** 4,261 of the 4,338 flagged pages (98%), and 98 of the top 100, have exactly 104 days since update. That looks like a bulk update or sync date, not 4,000 separate stories of neglect. This matches the MIXED staleness verdict: the gate filters on a batch marker more than on page-level neglect.
4. **Possible data gaps.** One page in the top 10 (#3) and two in the top 100 have zero clicks on big impressions. Those may be tracking holes, not weak titles.
5. **Young pages.** 34 of the top 100 are younger than 180 days. Refreshing a page that young may be premature.
6. **A pooled "typical" sits above the median.** Typical CTR is clicks over impressions, so big pages weigh most. At positions 3.1–6 it is 0.397%, but the median page there gets 0.279%. So an ordinary page can look like it is "missing" clicks.

If I had one more hour, I would rank within each client and score by the ratio of actual to typical clicks as well as the raw gap. I am not changing the rule now, because tuning it on these outcome numbers would make the baseline less honest.

In [ ]:
# Leakage check: no label-derived columns, no future window, no product flags in the rule
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "declining",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
print("Rule inputs:", RULE_INPUTS)
assert FORBIDDEN.isdisjoint(RULE_INPUTS), "a forbidden column is in the rule"

# the two derived inputs really do exclude the most recent 30 days (outcome window)
assert ((df["pre_impressions"] + df["impressions_last_30d"]) == df["impressions_90d"]).all()
assert ((df["pre_clicks"] + df["clicks_last_30d"]) == df["clicks_90d"]).all()
print("pre_impressions / pre_clicks exclude the last 30 days: OK")

print(f"Columns used by the rule: {len(RULE_INPUTS)} of {len(pd.read_csv(SOURCE, nrows=1).columns)} in the CSV")

# what a leak looks like: ranking by the label-derived trend_pct is almost perfect
leak_p100 = df.sort_values("trend_pct").head(100)["declining"].mean()
my_p100 = scored.head(100)["declining"].mean()
print(f"precision@100 ranking by trend_pct (a leak): {leak_p100:.3f}  |  my rule: {my_p100:.3f}")

Rule inputs: ['days_since_last_update', 'pre_impressions', 'pre_clicks', 'avg_position']
pre_impressions / pre_clicks exclude the last 30 days: OK
Columns used by the rule: 4 of 44 in the CSV
precision@100 ranking by trend_pct (a leak): 1.000  |  my rule: 0.570


**Leakage check: passed, with one honest note.**
- **No label-derived columns.** The rule never reads `trend_direction`, `trend_pct` or `is_declining_label` (checked in code). The `declining` outcome is only used to check signals and to score the ranking.
- **No future window.** The rule uses days 31–90 back. The most recent 30 days (the outcome window) are subtracted out, and the assertions confirm the arithmetic.
- **No product flags.** The starter CSV has no FlyRank flag columns, and the rule reads 4 of the 44 columns, all raw measurements.
- **The alarm test.** Ranking by the label-derived `trend_pct` gives precision@100 = 1.000. My rule gives 0.570. Perfect numbers are what a leak looks like, and mine are not that.
- **IDs are not features.** `content_id` only breaks ties and `client_id` only groups the checks. The typical-CTR table uses no labels and has no fitted weights, so a rule baseline needs no train/test split.
- **Honest note.** `avg_position` is a 90-day average, so its last 30 days overlap the outcome window. It is not part of how the outcome is computed (the outcome is built from impressions), but I can't cut position by window in this slice, so I treat any position effect as observed, not clean.

In [ ]:
VERDICTS = {"staleness": "MIXED", "ctr_vs_position": "CONFIRMED"}

metrics = {
    "notebook": "work/notebooks/w04_baseline_score.ipynb",
    "data": {"rows": int(len(df)), "outcome": "declining = trend_direction == 'down' (used only to check and score)",
             "base_rate_declining": round(float(base_rate), 4)},
    "rule": {"reason_code": REASON_CODE, "action_label": ACTION_LABEL, "inputs": RULE_INPUTS,
             "stale_days": STALE_DAYS, "volume_floor_impressions": VOLUME_FLOOR,
             "position_buckets": POS_LABELS,
             "typical_ctr_pct_by_position": {k: round(float(v) * 100, 3) for k, v in ctr_norm.items()}},
    "signals": {
        "staleness": {"verdict": VERDICTS["staleness"], "flag_linked": "refresh flags",
                      "n_stale": n_stale, "n_fresh": n_fresh,
                      "decline_rate_stale": round(float(rate_stale), 4), "decline_rate_fresh": round(float(rate_fresh), 4),
                      "pooled_gap": round(float(pooled_gap), 4), "clients_compared": int(len(within)),
                      "clients_stale_higher": stale_higher, "within_client_gap": round(within_gap, 4)},
        "ctr_vs_position": {"verdict": VERDICTS["ctr_vs_position"], "flag_linked": "CTR-fix logic",
                            "n_pages": int(len(known)), "decline_rate_by_ratio_bucket": {
                                str(k): round(float(v), 4) for k, v in ratio_table["declining_rate"].items()},
                            "n_by_ratio_bucket": {str(k): int(v) for k, v in ratio_table["n"].items()},
                            "clients_compared": int(len(within2)), "clients_low_higher": low_higher,
                            "within_client_gap": round(within2_gap, 4)},
    },
    "queue": {"rows": int(len(scored)), "flagged": int(len(flagged_rows)),
              "flagged_decline_rate": round(float(flagged_rate), 4), "rule_pool_decline_rate": round(float(pool_rate), 4)},
    "precision_at_k": {str(r["K"]): r["precision_at_K"] for r in prec_rows},
    "weak_picks": {"top10_declining": int(top10["declining"].sum()), "top100_declining": int(top100["declining"].sum()),
                   "top100_share_from_biggest_client": round(float(biggest_client_share), 3),
                   "top10_zero_click": zero_click_top10, "top100_zero_click": zero_click_top100,
                   "top100_same_update_day": same_day_top100, "top100_young_pages": young_top100},
    "leakage": {"trend_pct_precision_at_100": round(float(leak_p100), 3), "my_rule_precision_at_100": round(float(my_p100), 3)},
}
with open(OUT_DIR / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w04_baseline_metrics.json (small, committed receipts - no row-level data)")

Wrote work/outputs/w04_baseline_metrics.json (small, committed receipts - no row-level data)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, client URLs, or private queries anywhere (pages and clients appear only as short pseudonymous tags)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.